In [ ]:

import os
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2, 40).__str__()
import cv2


# hist = cv2.calcHist([img],[0],None, histSize=[255], ranges=[1,256])
# hist

In [ ]:
import math
imgFile = r"data/raw/test_slice_00300_CH2.jpg"
img = cv2.imread(imgFile, 0)
clipLimit = round(pow(math.e, - img.mean())*8,1)
clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=(25,25))
cl1 = clahe.apply(img)
# dst = cv2.equalizeHist(cl1)
cv2.imwrite(r'outputs/test_slice_00300_CH2_j.jpg',cl1)

# 图像局部二值化提高

In [ ]:
from pathlib import Path
from tqdm import tqdm

imgs = r"data/error"
imgAll = list(Path(imgs).glob("*.jpg"))

saveRoot = imgs + "_judge"
Path(saveRoot).mkdir(exist_ok=True)

for im in tqdm(imgAll):
    im_use = cv2.imread(str(im), 0)
    clahe = cv2.createCLAHE(clipLimit=7.0, tileGridSize=(25,25))
    cl1 = clahe.apply(im_use)
    cv2.imwrite(str(Path(saveRoot) / (im.stem+"_j.jpg")), cl1)

In [ ]:
import numpy as np

image_array = np.array(im)
black_pixels = np.where(np.sum(image_array, axis=0) == 0)
# left = np.min(black_pixels[1])

# top = np.min(black_pixels[0])

# right = np.max(black_pixels[1])
# bottom = np.max(black_pixels[0])

In [ ]:
def image_lighten(img):
    img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    # 设置自适应阈值
    clipLimit = round(pow(math.e, - img.mean())*10,1)
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=(25,25))
    # 获得结果
    cl1 = clahe.apply(img)
    cl1 = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    return cl1

def image_scale(img, scale=0.5):
    img_h, img_w = img.shape[0], img.shape[1]
    resize_h, resize_w = int(img_h*scale), int(img_w*scale)
    dst = cv2.resize(img, (resize_h, resize_w))
    return dst

# 裁剪需要的区域
1. 直接筛选明亮的区域，找到外轮廓
2. 运用中值滤波方法

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
cl1 = clahe.apply(img)
cv2.imwrite('clahe_2.jpg',cl1)

# 裁剪成1024*1024片段
## 裁剪图像

In [ ]:
import os
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2, 40).__str__()

from pathlib import Path
from tqdm import tqdm
import os
import math
import cv2

def img_histogram_adj(img, clipLimit=3.6, tileGridSize=(9,9)):
    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)
    cl1 = clahe.apply(img)
    return cl1

def crop_image(image_path, size=1024, step=620):
    image_path = str(image_path)
    # 打开图像
    image = cv2.imread(image_path, 0)
    image_width, image_height = image.shape[1], image.shape[0]
    
    # 计算裁剪后的图像数量
    num_images_horizontal = math.ceil(image_width / step)
    num_images_vertical = math.ceil(image_height / step)
    
    output_folder = image_path[:-4] + f"_j_{size}"
    name = os.path.basename(image_path)[:-4]
    
    # 创建输出文件夹
    os.makedirs(output_folder, exist_ok=True)
    
    cl_img = img_histogram_adj(image)
    cv2.imwrite(image_path[:-4] + "_j.jpg", cl_img)
    # 裁剪并保存小图像
    for i in range(num_images_vertical):
        for j in range(num_images_horizontal):
            # 计算裁剪位置
            left = j * step
            upper = i * step
            right = left + size
            lower = upper + size
            
            if (right > image_width) | (lower > image_height):
                continue
            
            # 裁剪图像
            output_image = cl_img[upper:lower, left:right]
            
            means, dev = cv2.meanStdDev(output_image)
            if means.ravel()[0] < 18:
                continue
            
            # 保存图像
            cv2.imwrite(os.path.join(output_folder, f'{name}_{left}_{upper}_{right}_{lower}.jpg'), output_image)


In [ ]:
# 多个数据夹的图像合并
import shutil

def concat_jpg(folders, saveroot):
    saveroot = Path(saveroot)
    saveroot.mkdir(exist_ok=True)
    for f in tqdm(folders):
        imgs = list(Path(f).glob("*.jpg"))
        [shutil.move(im, saveroot) for im in imgs]
    return print("Moved all")

foldersPath = r"data/normal_judge"
folders = list(f for f in Path(foldersPath).glob("*"))
saveRoot


# label转换

## 读取labelme文件

In [ ]:
import json

jf = r"data/labeldata\biaoji01\test_slice_00150_CH2_j_19220_11160_20244_12184.json"
with open(jf) as f:
    data = json.load(f)
    
data

### labelme 裁剪小片段

In [ ]:
import json
jsonFile = r"data/test_slice_01030_CH2_j.json"

with open(jsonFile) as f:
    data = json.load(f)
    
data

## labelme标记的聚合

In [ ]:
from pathlib import Path
import shutil
from tqdm import tqdm

def concat_labelme(dataset, saveRoot):
    saveRoot = Path(saveRoot)
    saveRoot.mkdir(exist_ok=True)
    dataset = Path(dataset)
    # imgs = list(dataset.glob("*.jpg"))
    jsons = list(dataset.glob("*.json"))
    for js in tqdm(jsons):
        im = dataset / (js.stem + ".jpg")
        if im.exists() is False:
            continue
        
        shutil.copy(str(im), str(saveRoot)) 
        shutil.copy(str(js), str(saveRoot))
    
    return print("moved all")

dataset = r"data/labeldata\biaoji04黄增丽--2\biaoji04"
saveRoot = r"data/labeldata\annotation04"
concat_labelme(dataset, saveRoot)

In [ ]:
im = Path(r"data/labeldata\test_slice_01250_CH2_j_1024\test_slice_01250_CH2_j_1024\、.json")
im.exists()

## json2mask

In [ ]:
# json to mask
import json
import numpy as np
import cv2

# imgFile = r"data/test_slice_01030_CH2_j.jpg"
# jsonFile = r"data/test_slice_01030_CH2_j.json"

# read json file
# with open(jsonFile, "r") as f:
#     data = json.load(f)

# # get the points
# n = len(data["shapes"])
# coords = [np.array(data["shapes"][i]["points"], dtype=np.int32) for i in range(n)]
# # points = data["shapes"][0]["points"]
# # points = np.array(points, dtype=np.int32)   # tips: points location must be int32

# # read image to get shape
# image = cv2.imread(imgFile)

# # create a blank image
# mask = np.zeros_like(image, dtype=np.uint8)

# # fill the contour with 255
# [cv2.fillPoly(mask, [ps], (255, 255, 255)) for ps in coords]

# # save the mask 
# cv2.imwrite(r"data/test_slice_01030_CH2_j_mask.jpg", mask)



In [ ]:
def json2mask_labelme(jsonFile):
    jsonFile = str(jsonFile)
    with open(jsonFile, "r") as f:
        data = json.load(f)

    # get the points
    n = len(data["shapes"])
    coords = [np.array(data["shapes"][i]["points"], dtype=np.int32) for i in range(n)]

    imgFile = jsonFile[:-5] + ".jpg"
    # read image to get shape
    image = cv2.imread(imgFile)

    # create a blank image
    mask = np.zeros_like(image, dtype=np.uint8)

    # fill the contour with 255
    [cv2.fillPoly(mask, [ps], (255, 255, 255)) for ps in coords]

    # save the mask 
    cv2.imwrite(imgFile[:-4] + ".png", mask)
    return



In [ ]:
dataset = r"data/labeldata\annatation01"
dataset = Path(dataset)

jsons = list(dataset.glob("*.json"))
for js in tqdm(jsons):
    json2mask_labelme(js)

## 将mask，json裁剪

In [ ]:
# mask, json 分割
from tqdm import tqdm
from pathlib import Path

def cut_imgAndMask(imgFile, MaskFile, savePath, size=1024, step=480):
    
    savePath = Path(savePath)
    
    # saveimg = savePath / "images"
    # savemsk = savePath / "masks"
    
    savePath.mkdir(exist_ok=True)
    # saveimg.mkdir(exist_ok=True)
    # savemsk.mkdir(exist_ok=True)
    
    # 背景图像
    img = cv2.imread(str(imgFile))
    msk = cv2.imread(str(MaskFile))
    img_h, img_w = msk.shape[0], msk.shape[1]
    
    img_wn = np.int32((img_w - size)/step)+1
    img_hn = np.int32((img_h - size)/step)+1
    
    for i in tqdm(range(img_wn)):
        for j in range(img_hn):
            x1 = int(step*i)
            y1 = int(step*j)
            x2 = x1+size
            y2 = y1+size
            dst    = img[y1:y2, x1:x2]
            dstMsk = msk[y1:y2, x1:x2]
            area = len(dstMsk[dstMsk==255])
            if area<10000:
                continue
            imgsavename = savePath / (Path(imgFile).stem + f"_{x1}_{y1}_{x2}_{y2}.jpg")
            msksavename = savePath / (Path(imgFile).stem + f"_{x1}_{y1}_{x2}_{y2}.png")
            cv2.imwrite(str(imgsavename), dst)
            cv2.imwrite(str(msksavename), dstMsk)
    return



In [ ]:
imgFile = r"data/test_slice_01030_CH2_j.jpg"
mskFile = r"data/test_slice_01030_CH2_j_mask.jpg"
savePath = r"data/test_slice_01030_test"

cut_imgAndMask(imgFile, mskFile, savePath)

In [ ]:
# # mask to json
# import os
# os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2, 40).__str__()
# import cv2
# import json

# imgFile = r"data/test_slice_01030_CH2_j_14880_14400_15904_15424_mask.jpg"
# img = cv2.imread(imgFile, 0)

# _, binary = cv2.threshold(img,10,255,cv2.THRESH_BINARY)
# contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# dic = {
#     "version": "5.2.1", 
#     "flags": {},
#     "shapes":list(), 
#     "imagePath":os.path.basename(imgFile),
#     "imageData": None,
#     "imageHeight":img.shape[0], 
#     "imageWidth":img.shape[1]}

# for contour in contours:
#         # temp = list()
#         # for point in contour[2:]:
#         #     if (len(temp) > 1) and (temp[-2][0] * temp[-2][1] * int(point[0][0]) * int(point[0][1]) != 0) and (int(point[0][0]) - temp[-2][0]) * (
#         #                     temp[-1][1] - temp[-2][1]) == (int(point[0][1]) - temp[-2][1]) * (temp[-1][0] - temp[-1][0]):
#         #         temp[-1][0] = int(point[0][0])
#         #         temp[-1][1] = int(point[0][1])
#         #     else:
#         #         temp.append([int(point[0][0]), int(point[0][1])])
#         # if len(temp) < 2:
#         #     continue
        
#         if len(contour) < 2:
#             continue
#         # 需要进行轮廓线简化
#         cont = np.array(contour).reshape(-1,1,2)
        
        
#         epsilon = 0.01*cv2.arcLength(cont, True)
#         approx = cv2.approxPolyDP(cont, epsilon, True)
#         approx = approx.reshape(-1,2).tolist()
#         dic["shapes"].append({"label": "duct", "points":approx, "group_id": None, "description": "",
#                                 "shape_type": "polygon", "flags": {}})

# with open(imgFile[:-4] + ".json", mode='w', encoding="utf-8") as f:
#     json.dump(dic, f)

## mask2json

In [ ]:
# mask to json
import os
os.environ["OPENCV_IO_MAX_IMAGE_PIXELS"] = pow(2, 40).__str__()
import cv2
import json

def mask2json_labelme(maskFile):
    maskFile = str(maskFile)
    img = cv2.imread(str(maskFile), 0)

    _, binary = cv2.threshold(img,10,255,cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    dic = {
        "version": "5.2.1", 
        "flags": {},
        "shapes":list(), 
        "imagePath":os.path.basename(maskFile),
        "imageData": None,
        "imageHeight":img.shape[0], 
        "imageWidth":img.shape[1]}

    for contour in contours:
        # 需要进行轮廓线简化
        cont = np.array(contour).reshape(-1,1,2)
        epsilon = 0.01*cv2.arcLength(cont, True)
        area = cv2.contourArea(cont)
        if area<32*32:
            continue
        approx = cv2.approxPolyDP(cont, epsilon, True)
        approx = approx.reshape(-1,2).tolist()
        dic["shapes"].append({"label": "duct", "points":approx, "group_id": None, "description": "",
                                "shape_type": "polygon", "flags": {}})

    with open(maskFile[:-4] + ".json", mode='w', encoding="utf-8") as f:
        json.dump(dic, f)
    return

In [ ]:
mskRoot = r"data/medicineAug_train01\masks"
msks = list(Path(mskRoot).glob("*.png"))


In [ ]:
for m in tqdm(msks):
    mask2json_labelme(m)

In [ ]:
mskFile = r"data/test_slice_01030_CH2_j_14880_18720_15904_19744_mask.jpg"
mask2json_labelme(mskFile)

## 数据增广

In [ ]:
import imageio
import cv2
import numpy as np
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage

imgPath = r"data/test_slice_01030_test\masks\test_slice_01030_CH2_j_14880_10560_15904_11584.jpg"
mskPath = r"data/test_slice_01030_test\masks\test_slice_01030_CH2_j_14880_10560_15904_11584.png"

image  = cv2.imread(imgPath)
msk = cv2.imread(mskPath,0)
# 需要指定类别0,1,...
segmap = np.zeros((image.shape[0], image.shape[1]), dtype=np.int32)
segmap[msk>0] = 1

segmap = SegmentationMapsOnImage(segmap, shape=image.shape)

seq = iaa.Sequential([
    iaa.Dropout([0.05, 0.2]),      # drop 5% or 20% of all pixels
    iaa.Sharpen((0.0, 1.0)),       # sharpen the image
    iaa.Affine(rotate=(-45, 45)),  # rotate by -45 to 45 degrees (affects segmaps)
    iaa.ElasticTransformation(alpha=50, sigma=5)  # apply water effect (affects segmaps)
], random_order=True)

# Augment images and segmaps.
images_aug = []
segmaps_aug = []
for _ in range(5):
    images_aug_i, segmaps_aug_i = seq(image=image, segmentation_maps=segmap)
    images_aug.append(images_aug_i)
    segmaps_aug.append(segmaps_aug_i)

cells = []
for image_aug, segmap_aug in zip(images_aug, segmaps_aug):
    cells.append(image)                                         # column 1
    cells.append(segmap.draw_on_image(image)[0])                # column 2
    cells.append(image_aug)                                     # column 3
    cells.append(segmap_aug.draw_on_image(image_aug)[0])        # column 4
    cells.append(segmap_aug.draw(size=image_aug.shape[:2])[0])  # column 5

grid_image = ia.draw_grid(cells, cols=5)
imageio.imwrite(r"data/test_slice_01030_CH2_j_14880_10560_15904_11584_ex.jpg", grid_image)

In [ ]:
import imageio
import cv2
import numpy as np
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
import random
from pathlib import Path
from tqdm import tqdm

def augument_mask(imgFile, mskFile):
    image  = cv2.imread(str(imgFile))
    msk = cv2.imread(str(mskFile),0)
    # 需要指定类别0,1,...,这里只有1类 duct
    segmap = np.zeros((image.shape[0], image.shape[1]), dtype=bool)
    segmap[msk>10] = True
    segmap = SegmentationMapsOnImage(segmap, shape=image.shape)

    seq = iaa.SomeOf((3, 8),[
        iaa.Dropout([0.04, 0.12]),      # drop 5% or 20% of all pixels
        iaa.Sharpen((0.0, 1.0)),       # sharpen the image
        iaa.Add((-10, 40)),
        iaa.ScaleX((0.8, 1.2)),
        iaa.ScaleY((0.8, 1.2)),
        # iaa.Affine(rotate=(-30, 30)),  # rotate by -45 to 45 degrees (affects segmaps)
        iaa.Affine(rotate=(-45, 45), 
                   shear={"x": (-7, 7), "y": (-7, 7)}, 
                   translate_percent={"x": (-0.15, 0.15), "y": (-0.15, 0.15)}), # 随机旋转+剪切变换
        iaa.PiecewiseAffine(scale=(0.01, 0.05)),
        iaa.Rot90([1, 3]),
        # iaa.AdditiveGaussianNoise(scale=0.01*255), # 添加高斯噪声点
        # iaa.RandAugment(n=(0, 2), m=(0, 3)), # 增加随机图像变换
        # iaa.imgcorruptlike.ShotNoise(severity=1), # 散粒噪声
        # iaa.Crop(px=(0, 16)), # 随机裁剪
        # iaa.Cutout(nb_iterations=(2,5),size=0.05, cval=0),
        iaa.CropAndPad(percent=(-0.15, 0.15)),
        iaa.Fliplr(0.5), # 随机水平翻转
        iaa.Flipud(0.5)
        ], random_order=True)
    
    image_aug, segmap_aug = seq(image=image, segmentation_maps=segmap)
    return image_aug, segmap_aug.draw(size=image_aug.shape[:2])[0]

def mask_folderAug(dataset,  fused = True, rate=1):
    # random.seed(12)
    dataset = Path(dataset)
    imgPaths = list(Path(dataset).glob('*.jpg'))
    mskPaths = list(Path(dataset).glob('*.png'))
    
    if fused is False:
        augSave = dataset.parent / (dataset.stem + "_aug")
        Path(augSave).mkdir(exist_ok=True)
    else:
        augSave = dataset
    
    pbar = tqdm(total=len(imgPaths))
    for img in imgPaths:
        r = random.random()
        if r>rate:
            continue
        msk = dataset / (img.stem + ".png")
        image_aug, segmap_aug = augument_mask(str(img), str(msk))
        
        cv2.imwrite(str(augSave / ("aug_" + img.name)), image_aug)
        cv2.imwrite(str(augSave / ("aug_" + msk.name)), segmap_aug)
        pbar.update(1)
    return print("Finished")


In [ ]:
dataset = r"data/labeldata\annatation02"
mask_folderAug(dataset)

In [ ]:
image = ia.quokka(size=(128, 128), extract="square")
image.shape

In [ ]:
imgPath = r"data/test_slice_01030_test\masks\test_slice_01030_CH2_j_14880_10560_15904_11584.jpg"
mskPath = r"data/test_slice_01030_test\masks\test_slice_01030_CH2_j_14880_10560_15904_11584.png"

# image  = cv2.imread(imgPath)
segmap = cv2.imread(mskPath,0)
segmap.shape

# labelme-增广

In [ ]:
import imageio
import cv2
import numpy as np
import imgaug as ia
import imgaug.augmenters as iaa
from imgaug.augmentables.segmaps import SegmentationMapsOnImage
from pathlib import Path
from tqdm import tqdm
import json

def labelme_augment(imgPath, jsonPath, savePath, classes=["car", "truck"]):
    # 读取labelme json
    with open(jsonPath, "r") as f:
        data = json.load(f)
    
    # 创建保存文件夹
    savePath = Path(savePath)
    savePath.mkdir(exist_ok=True)
    
    img_name = data["imagePath"]
    n = len(data["shapes"])
    polys = [(data["shapes"][i]["label"], data["shapes"][i]['points']) for i in range(n)]
    if len(polys)==0:
        return
    
    image  = cv2.imread(str(imgPath))
    segmap = np.zeros((image.shape[0], image.shape[1], 1), dtype=np.int32)

    for i in range(len(classes)):
        mask = np.zeros((image.shape[0], image.shape[1]), dtype=np.uint8)
        t = classes[i]
        polys_i = [poly for lab, poly in polys if lab==t]
        res_polys = []
        for pts in polys_i:
            pts = np.array(pts, np.int32)
            pts = pts.reshape((1, -1, 2))
            res_polys.append(pts)
        mask = cv2.fillPoly(mask, res_polys, (255))
        segmap[mask>0] = i+1

    segmap = SegmentationMapsOnImage(segmap, shape=image.shape)
    
    sometimes = lambda aug: iaa.Sometimes(0.5, aug)
    
    seq = iaa.Sequential(
        [
            iaa.Fliplr(0.5), # 随机水平翻转
            iaa.Flipud(0.5),
            iaa.Affine(rotate=(10, 80)),
            sometimes(iaa.Crop(percent=(0, 0.1))),
            
            iaa.SomeOf((2, 7),
                [
                    iaa.Dropout([0.04, 0.12]),      # drop 5% or 20% of all pixels
                    iaa.Sharpen((0.0, 1.0)),
                    iaa.Rot90([1, 3]),
                    iaa.CropAndPad(percent=(-0.15, 0.15)),
                    iaa.PiecewiseAffine(scale=(0.01, 0.05)),
                    iaa.Add((-10, 50)),
                    iaa.ScaleX((0.8, 1.2)),
                    iaa.ScaleY((0.8, 1.2))
                ], random_order=True)
        ])
    image_aug, segmap_aug = seq(image=image, segmentation_maps=segmap) # 生成增广后的图像和mask矩阵
    # 储存生成的结果
    imgsave = savePath / ("aug_"+ img_name)
    cv2.imwrite(str(imgsave), image_aug)
    jsonsave = savePath / ("aug_"+img_name[:-4]+".json")
    dic = {
        "version": "5.2.1", 
        "flags": {},
        "shapes":list(), 
        "imagePath":imgsave.name,
        "imageData": None,
        "imageHeight":image.shape[0], 
        "imageWidth":image.shape[1]}
    # 对于mask矩阵 找到对应的外轮廓json
    segarr = segmap_aug.get_arr()
    for i in range(len(classes)):
        t = classes[i]
        seg_i = (segarr[:,:,0]==(i+1))
        find_seg = seg_i.astype(np.uint8)*255
        contours, _ = cv2.findContours(find_seg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            # 需要进行轮廓线简化
            cont = np.array(contour).reshape(-1,1,2)
            epsilon = 0.01*cv2.arcLength(cont, True)
            area = cv2.contourArea(cont)
            if area<32*32:
                continue
            approx = cv2.approxPolyDP(cont, epsilon, True)
            approx = approx.reshape(-1,2).tolist()
            dic["shapes"].append({"label": t, "points":approx, "group_id": None, "description": "",
                                    "shape_type": "polygon", "flags": {}})
    with open(jsonsave, mode='w', encoding="utf-8", newline='\n') as f:
        json.dump(dic, f)
    return




In [ ]:
classes = ["duct"]

jsons = list(Path(r"data/labeldata\annotation04").glob("*.json"))
savePath = r"data/labeldata\annotation04"
# jsons


In [ ]:
for j in tqdm(jsons):
    jsonFile = str(j)
    imgFile = jsonFile[:-5] + ".jpg"
    labelme_augment(imgFile, jsonFile, savePath, classes=classes)

# 转换成YOLO-seg

In [ ]:
from pathlib import Path
from tqdm import tqdm
import shutil
import json
import numpy as np

def labelme2yoloseg(jsonFile, t={"duct": 0}):
    with open(jsonFile) as f:
        data = json.load(f)
    img_w = data['imageWidth']
    img_h = data['imageHeight']
    
    n = len(data["shapes"])
    if n == 0:
        return None

    res = []
    for i in range(n):
        c = data["shapes"][i]['label']
        poly = np.array(data["shapes"][i]['points'])
        poly_yolo = poly / np.array([img_w, img_h])
        r = str(t[c]) + " " + " ".join([(f"{coord[0]} {coord[1]}") for coord in poly_yolo])
        res.append(r)
    return res

def covert_labelmeFolder2yolo(folderPath, saveRoot):
    folder = Path(folderPath)
    saveRoot = Path(saveRoot)
    
    labelRoot = saveRoot / "labels"
    imageRoot = saveRoot / "images"
    
    saveRoot.mkdir(exist_ok=True)
    labelRoot.mkdir(exist_ok=True)
    imageRoot.mkdir(exist_ok=True)
    
    files = folder.glob("*.json")
    for fPath in tqdm(files):
        imgPath = folder / (fPath.stem + ".jpg")
        assert imgPath.exists(), "img error"
        
        yolosegs = labelme2yoloseg(fPath)
        if yolosegs is None:
            continue
        shutil.copy(str(imgPath), imageRoot)
        
        labelPath = labelRoot / (fPath.stem + ".txt")
        with open(labelPath, "w") as f:
            f.write("\n".join(yolosegs))
    return


In [ ]:
folderPath = r"data/labeldata\annotation04"
saveRoot = r"data/labeldata\medAug_annatation04"

covert_labelmeFolder2yolo(folderPath, saveRoot)

### 划分训练集测试集

In [ ]:
import shutil
from pathlib import Path
from collections import Counter
from tqdm import tqdm

import yaml
import numpy as np
import pandas as pd

import os
os.environ['NUMEXPR_MAX_THREADS'] = '16'

from sklearn.model_selection import train_test_split

def yolo_train_val(datasetPath, classes, test_size=0.15, random_state=30):
    # 生成训练集测试集

    dataset_path = Path(datasetPath) # replace with 'path/to/dataset' for your custom data
    labels = sorted(dataset_path.glob("labels/*.txt")) # all data in 'labels'
    indx = [l.stem for l in labels] # uses base filename as ID (no extension)
    labels_df = pd.DataFrame([], dtype='float64', columns=range(len(classes)), index=indx)
    
    # 统计 label 数量
    for label in labels:
        lbl_counter = Counter()

        with open(label,'r') as lf:
            lines = lf.readlines()

        for l in lines:
            lbl_counter[int(l.split(' ')[0])] += 1

        labels_df.loc[label.stem] = lbl_counter
    labels_df = labels_df.dropna(axis=0, how="all")
    labels_df = labels_df.fillna(0) # replace `nan` values with `0.0`
    labels_df.columns = classes
    
    train_set, val_set = train_test_split(labels_df, shuffle=True, test_size=test_size, random_state=random_state)
    for t in classes:
        print(f"{t :<15} |: train: {int(train_set[t].sum()) :<7}, val: {int(val_set[t].sum()) :<7}")
    
    # 创建保存文件夹
    save_path = Path(dataset_path.parent / f'{dataset_path.stem}_yolo')
    save_path.mkdir(parents=True, exist_ok=True)
    (save_path / 'images'/ 'train' ).mkdir(parents=True, exist_ok=True)
    (save_path / 'labels'/ 'train' ).mkdir(parents=True, exist_ok=True)
    (save_path / 'images'/ 'val' ).mkdir(parents=True, exist_ok=True)
    (save_path / 'labels'/ 'val' ).mkdir(parents=True, exist_ok=True)

    # 写入yaml文件
    dataset_yaml = save_path / f'{save_path.stem}.yaml'

    with open(dataset_yaml, 'w') as ds_y:
        yaml.safe_dump({
            'path': Path(".").joinpath(save_path.stem).as_posix(),
            'train': 'images/train',
            'val': 'images/val',
            'nc' : len(classes),
            'names': classes
        }, ds_y)
    
    trains = train_set.index.to_list()
    with tqdm(total=len(trains)) as pbar:
        pbar.set_description('trains ')
        for data in trains:
            shutil.copy(dataset_path / 'images' / (data + ".jpg"), save_path / 'images'/ 'train')
            shutil.copy(dataset_path / 'labels' / (data + ".txt"), save_path / 'labels'/ 'train')
            pbar.update(1)
    
    vals = val_set.index.to_list()
    with tqdm(total=len(vals)) as pbar:
        pbar.set_description('vals ')
        for data in vals:
            shutil.copy(dataset_path / 'images' / (data + ".jpg"), save_path / 'images'/ 'val')
            shutil.copy(dataset_path / 'labels' / (data + ".txt"), save_path / 'labels'/ 'val')
            pbar.update(1)
    
    return print("Finished")

classes = ["duct"]

# 生成训练集测试集
datasetPath = r"data/labeldata\medAug_annatation04"

yolo_train_val(datasetPath, classes)



In [ ]:
t={"duct": 0}

In [ ]:
# import package
import labelme2coco
from pathlib import Path
# set directory that contains labelme annotations and image files
labelme_folder = r"data/labeldata\train_01"

# set export dir
export_dir = r"data/labeldata\train_01_coco"
Path(export_dir).mkdir(exist_ok=True)
# set train split rate
train_split_rate = 0.85

# convert labelme annotations to coco
labelme2coco.convert(labelme_folder, export_dir, train_split_rate)

# 统计有多少个

In [ ]:
import json
from pathlib import Path
from tqdm import tqdm

def countlabelnum(annotationRoot):
    dataset = Path(annotationRoot)
    # imgs = list(dataset.glob("*.jpg"))
    jsons = list(dataset.glob("*.json"))
    count = 0
    for js in tqdm(jsons):
        im = dataset / (js.stem + ".jpg")
        if im.exists() is False:
            continue
        with open(js) as f:
            data = json.load(f)
        n = len(data["shapes"])
        count += n
    return print(f"Have {count} labels")
countlabelnum(r"data/scaled_images")